# Colab 02 — MobileNetV2: S1 → S2 → S3, model lengkap (paket model akhir R15)

**Cara menjalankan**
1. *Runtime > Ubah jenis runtime > **L4 GPU*** (notebook berhenti bila GPU lain, agar ketiga backbone dilatih di jenis GPU yang sama).
2. *Runtime > **Run all***. Izinkan akses Google Drive saat diminta.
3. Bila runtime terputus, batas sesi habis, atau dihentikan sengaja: sambungkan lagi, pilih L4, lalu *Run all*. Semua yang sudah selesai dilewati dan training dilanjutkan dari epoch terakhir yang tersimpan di Drive.

Hasil tersimpan di **My Drive/TA_R15_hasil/outputs** milik akun ini. Setelah selesai, unduh folder itu ke komputer lokal untuk digabung (notebook 04) dan diukur latensinya (notebook 05).

Alur setiap skenario: **CV 5 fold → E\* → model final CNN → holdout CNN → model lengkap (specialist + fusi + kunci VT)**. Setelah S3: ringkasan, uji efek, dan XAI.

**Aturan yang mengikat:**
- **Metode terkunci.** Hash protokol data, hash metode CNN, dan hash metode S3 dicek di sel 3 dan harus sama dengan MA1 (R15). Jangan mengubah berkas di folder bundel.
- **Holdout hanya untuk evaluasi akhir.** Holdout dievaluasi satu kali per model final dan tidak dipakai untuk memilih apa pun (E\* dipilih dari CV development).
- **Lanjut otomatis.** Setiap sel aman dijalankan ulang. Fold/model yang lengkap diverifikasi lalu dipakai ulang, training yang terputus dilanjutkan dari `last.pt` (disimpan tiap epoch), holdout dan model lengkap yang sudah selesai hanya dibaca ulang.
- **Tidak ada yang dihapus atau ditimpa.** Bila suatu hasil sudah ada tetapi masukannya berbeda, sel berhenti dengan pesan, bukan menimpa.


In [ ]:
# Sel 1: mount Drive, pasang paket, salin bundel ke disk lokal runtime (sekitar 5-10 menit pada sesi pertama)
ARCH = "mobilenet_v2"
from google.colab import drive
drive.mount("/content/drive")
import os, sys
from pathlib import Path
# Fitur ritme lokal (E1-E4) dihitung paralel per rekaman; 4 worker mencegah RAM habis pada rekaman AFDB 10 jam.
os.environ.setdefault("V4_RHYTHM_WORKERS", "4")
MYDRIVE = Path("/content/drive/MyDrive")
CARI = [MYDRIVE, MYDRIVE.parent / ".shortcut-targets-by-id"]   # akun 2/3: pintasan ke folder bersama
BUNDEL = next((p.parent for root in CARI for d in range(4) for p in sorted(root.glob("/".join(["*"] * d + ["BUNDEL_R15.json"])))), None)
assert BUNDEL, ("Folder TA_R15_COLAB tidak ditemukan. Akun 1: unggah foldernya ke My Drive. Akun 2/3: buka 'Dibagikan kepada saya', "
                "klik kanan TA_R15_COLAB > Atur > Tambahkan pintasan ke Drive > My Drive, lalu Run all lagi.")
print("Bundel:", BUNDEL)
sys.path.insert(0, str(BUNDEL))
sys.dont_write_bytecode = True   # jangan menulis __pycache__ ke folder bundel (akun 2/3 hanya Pelihat)
import colab_setup as C
C.pasang_paket()
INFO = C.siapkan(ARCH, BUNDEL)


In [ ]:
import sys, json, time
from pathlib import Path
FOLDER = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v4_pipeline.py").exists() or (p / "eksperimen_v4_svdb" / "v4_pipeline.py").exists())
FOLDER = FOLDER if (FOLDER / "v4_pipeline.py").exists() else FOLDER / "eksperimen_v4_svdb"
sys.path.insert(0, str(FOLDER))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
import v4_runner as R
import v4_figures as F
print("Folder eksperimen:", FOLDER)
import v4_model_lengkap as M


In [ ]:
# Sel 3: lingkungan + integritas. Hash berkas rilis; kodenya setara dengan run resmi (d7f73001 / 44315f01), lihat scripts/verify_hashes.py.
import platform, torch, shap
import v4_pipeline as v4, v4_s3
p = R.protocol()
EXPECTED = {
 "data_protocol_hash": "9db60710fd60433663bab15e1bbe3f61ba0b97cdf0ef59f0fc6531d5d510735d",
 "method_hash": "5a131004958b6722980e464a5da2ecf195db23970119a04a7f369a711cbaee81",
 "s3_method_hash": "8bb5f1d88a8ae250add56c8f0b641076e269d9e55718d450fe66cced57a5d753"
}
current = {"data_protocol_hash": p.hash, "method_hash": v4.method_hash(), "s3_method_hash": v4_s3.s3_method_hash()}
for key in EXPECTED:
    assert current[key] == EXPECTED[key], f"{key} berbeda: {current[key]}. BERHENTI, jangan lanjut training."
env = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0), "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
       "shap": shap.__version__, "cpu": platform.processor(), "os": platform.platform(), "arch": ARCH, **INFO}
(p.out / f"env_colab_{ARCH}.json").write_text(json.dumps(env, indent=2)); display(pd.Series(current)); env


In [ ]:
# Sel 4: pemeriksaan data (jumlah window/pasien, tidak ada overlap pasien development-holdout)
from ecgtl.hierarchical.data import read_manifest
cfg = p.cfg["data"]
raw = read_manifest(p.prepared / "global_manifest.csv")
dev, hold = F.manifest("development"), F.manifest("holdout")
checks = pd.DataFrame([
    ("window manifest", len(raw), cfg["expected_windows"]),
    ("pasien total", raw.patient_id.nunique(), cfg["expected_patients"]),
    ("pasien development", dev.patient_id.nunique(), cfg["expected_development_patients"]),
    ("pasien holdout", hold.patient_id.nunique(), cfg["expected_holdout_patients"]),
    ("window development (label_valid)", len(dev), 16167),
    ("window holdout (label_valid)", len(hold), 3852),
], columns=["item", "aktual", "diharapkan"])
assert (checks.aktual == checks.diharapkan).all(), checks
assert not set(dev.patient_id) & set(hold.patient_id), "Overlap pasien development-holdout"
assert dev.groupby("patient_id").fold.nunique().max() == 1, "Pasien muncul di lebih dari satu fold"
checks


In [ ]:
# Sel 5: cache embedding ImageNet beku untuk S1/S2 (dibuat sekali, lalu diarsipkan ke Drive agar sesi berikutnya cepat)
t = time.time(); print(R.experiment(ARCH, "S1").base.cache_features().to_dict("records"), f"{time.time() - t:.0f} dtk")
C.simpan_cache(ARCH, p.hash)


In [ ]:
# Sel 6: sampel XAI dibekukan dari label saja (identik di ketiga akun; dicek saat penggabungan)
import v4_xai as X
samples = X.freeze_samples(); print("sha256 sampel XAI:", samples["sha256"])
R.status(ARCH)


## S1 — Backbone ImageNet **beku**, **tanpa** class weighting di semua tahap: CNN τ=1, specialist dan fusi tanpa bobot.


### S1.a CV 5 fold (CNN)
Fold yang sudah selesai dipakai ulang. Bila runtime terputus, *Run all* lagi dan training dilanjutkan dari `last.pt`. Progres per epoch juga ditulis di `outputs/logs/<ARCH>_S1.log`.


In [ ]:
cv_S1 = R.cv(ARCH, "S1"); cv_S1


### S1.b Epoch final bersama E\*
E\* = argmax rerata Macro-F1 validasi pada semua epoch yang tersedia di kelima fold (receipt `selection/`).


In [ ]:
sel = R.final_epoch(ARCH, "S1"); summary = R.cv_summary(ARCH, "S1")
print("E* =", sel["selected_epoch"], "| rerata Macro-F1 CNN di E* =", round(sel["selected_mean_macro_f1"], 4))
display(pd.read_csv(R.protocol().out / "tables" / ARCH / "S1_cv_folds.csv").round(3))


### S1.c Model final CNN
Dilatih pada seluruh data development selama E\* epoch.


In [ ]:
final_S1 = R.final_model(ARCH, "S1"); {k: final_S1[k] for k in ["epochs_run", "training_s", "trainable_parameters", "checkpoint_sha256"]}


### S1.d Holdout CNN (komponen, sekali)
Probabilitas CNN model final pada holdout. Bila dijalankan ulang, hanya dibaca dari receipt.


In [ ]:
h = R.holdout(ARCH, "S1")
print(f"CNN saja, Macro-F1 holdout {h['macro_f1']:.4f}")


### S1.e Model lengkap: CNN + Beat-Level Specialist + fusi + kunci VT
Resep MA1 (`v4_model_lengkap.py`). Development: fusi nested per fold luar. Holdout: fusi dilatih pada seluruh development. Beberapa menit, tanpa GPU. Hasil di `outputs/model_lengkap/<ARCH>/S1/`.


In [ ]:
ml_S1 = M.jalankan(ARCH, "S1")
display(pd.DataFrame({k: ml_S1[k] for k in ["dev_cnn_saja", "dev_lengkap", "holdout_cnn_saja", "holdout_lengkap"]}).T.round(4))


## S2 — Backbone ImageNet **beku** dengan **class weighting** di semua tahap: CNN τ=2, specialist dan fusi berbobot (1/frekuensi)^0,5. Untuk EfficientNet-B0 ini resep MA1.


### S2.a CV 5 fold (CNN)
Fold yang sudah selesai dipakai ulang. Bila runtime terputus, *Run all* lagi dan training dilanjutkan dari `last.pt`. Progres per epoch juga ditulis di `outputs/logs/<ARCH>_S2.log`.


In [ ]:
cv_S2 = R.cv(ARCH, "S2"); cv_S2


### S2.b Epoch final bersama E\*
E\* = argmax rerata Macro-F1 validasi pada semua epoch yang tersedia di kelima fold (receipt `selection/`).


In [ ]:
sel = R.final_epoch(ARCH, "S2"); summary = R.cv_summary(ARCH, "S2")
print("E* =", sel["selected_epoch"], "| rerata Macro-F1 CNN di E* =", round(sel["selected_mean_macro_f1"], 4))
display(pd.read_csv(R.protocol().out / "tables" / ARCH / "S2_cv_folds.csv").round(3))


### S2.c Model final CNN
Dilatih pada seluruh data development selama E\* epoch.


In [ ]:
final_S2 = R.final_model(ARCH, "S2"); {k: final_S2[k] for k in ["epochs_run", "training_s", "trainable_parameters", "checkpoint_sha256"]}


### S2.d Holdout CNN (komponen, sekali)
Probabilitas CNN model final pada holdout. Bila dijalankan ulang, hanya dibaca dari receipt.


In [ ]:
h = R.holdout(ARCH, "S2")
print(f"CNN saja, Macro-F1 holdout {h['macro_f1']:.4f}")


### S2.e Model lengkap: CNN + Beat-Level Specialist + fusi + kunci VT
Resep MA1 (`v4_model_lengkap.py`). Development: fusi nested per fold luar. Holdout: fusi dilatih pada seluruh development. Beberapa menit, tanpa GPU. Hasil di `outputs/model_lengkap/<ARCH>/S2/`.


In [ ]:
ml_S2 = M.jalankan(ARCH, "S2")
display(pd.DataFrame({k: ml_S2[k] for k in ["dev_cnn_saja", "dev_lengkap", "holdout_cnn_saja", "holdout_lengkap"]}).T.round(4))


## S3 — **Fine-tuning penuh** encoder dan head, diinisialisasi dari checkpoint S2 fold yang sama (τ=2), specialist dan fusi berbobot. Skenario terlama; setiap epoch disimpan ke Drive.


### S3.a CV 5 fold (CNN)
Fold yang sudah selesai dipakai ulang. Bila runtime terputus, *Run all* lagi dan training dilanjutkan dari `last.pt`. Progres per epoch juga ditulis di `outputs/logs/<ARCH>_S3.log`.

Sebelum fine-tuning tiap fold, audit inisialisasi memeriksa bahwa head = checkpoint S2, encoder = ImageNet, jalur cache mereproduksi probabilitas S2, dan jalur citra mentah hanya berbeda derau numerik (`initialization_audit_r15.json`).


In [ ]:
cv_S3 = R.cv(ARCH, "S3"); cv_S3


### S3.b Epoch final bersama E\*
E\* = argmax rerata Macro-F1 validasi pada semua epoch yang tersedia di kelima fold (receipt `selection/`).


In [ ]:
sel = R.final_epoch(ARCH, "S3"); summary = R.cv_summary(ARCH, "S3")
print("E* =", sel["selected_epoch"], "| rerata Macro-F1 CNN di E* =", round(sel["selected_mean_macro_f1"], 4))
display(pd.read_csv(R.protocol().out / "tables" / ARCH / "S3_cv_folds.csv").round(3))


### S3.c Model final CNN
Dilatih pada seluruh data development selama E\* epoch.


In [ ]:
final_S3 = R.final_model(ARCH, "S3"); {k: final_S3[k] for k in ["epochs_run", "training_s", "trainable_parameters", "checkpoint_sha256"]}


### S3.d Holdout CNN (komponen, sekali)
Probabilitas CNN model final pada holdout. Bila dijalankan ulang, hanya dibaca dari receipt.


In [ ]:
h = R.holdout(ARCH, "S3")
print(f"CNN saja, Macro-F1 holdout {h['macro_f1']:.4f}")


### S3.e Model lengkap: CNN + Beat-Level Specialist + fusi + kunci VT
Resep MA1 (`v4_model_lengkap.py`). Development: fusi nested per fold luar. Holdout: fusi dilatih pada seluruh development. Beberapa menit, tanpa GPU. Hasil di `outputs/model_lengkap/<ARCH>/S3/`.


In [ ]:
ml_S3 = M.jalankan(ARCH, "S3")
display(pd.DataFrame({k: ml_S3[k] for k in ["dev_cnn_saja", "dev_lengkap", "holdout_cnn_saja", "holdout_lengkap"]}).T.round(4))


## Ringkasan MobileNetV2
Model lengkap S1/S2/S3. Dua efek dibaca: S2−S1 (class weighting) dan S3−S2 (fine-tuning).


In [ ]:
# Ringkasan model lengkap S1/S2/S3 + confusion matrix holdout
import numpy as np
rows = {}
for sc in ("S1", "S2", "S3"):
    r = M.jalankan(ARCH, sc)
    rows[f"{sc} dev (CV)"] = {"E*": r["E*"], **r["dev_lengkap"]}
    rows[f"{sc} holdout"] = {"E*": r["E*"], **r["holdout_lengkap"]}
tabel = pd.DataFrame(rows).T; display(tabel.round(4))
folder = R.protocol().out / "model_lengkap" / ARCH
if not (folder / "ringkasan_S1_S2_S3.csv").exists():
    tabel.to_csv(folder / "ringkasan_S1_S2_S3.csv")
fig, axes = plt.subplots(1, 3, figsize=(19, 6.2))
for ax, sc in zip(axes, ("S1", "S2", "S3")):
    cm = np.load(folder / sc / "confusion_holdout.npy"); rec = cm / cm.sum(1, keepdims=True)
    ax.imshow(rec, cmap="Blues", vmin=0, vmax=1)
    for i in range(5):
        for j in range(5):
            ax.text(j, i, f"{int(cm[i, j])}\n{rec[i, j]:.2f}", ha="center", va="center", color="white" if rec[i, j] > .6 else "black")
    f1 = 2 * np.diag(cm) / (cm.sum(0) + cm.sum(1))
    ax.set_xticks(range(5), M.C); ax.set_yticks(range(5), M.C); ax.set_xlabel("Prediksi"); ax.set_ylabel("Label")
    ax.set_title(f"{sc} holdout | Macro-F1 {f1.mean():.3f} | F1 S {f1[1]:.3f} | Recall S {rec[1, 1]:.3f}")
fig.tight_layout()
if not (folder / "confusion_holdout_S1_S2_S3.png").exists():
    fig.savefig(folder / "confusion_holdout_S1_S2_S3.png", dpi=130)
plt.show()


In [ ]:
# Efek class weighting (S2 - S1) dan fine-tuning (S3 - S2): bootstrap per pasien holdout, 2.000 ulangan, koreksi Holm
for a, b in (("S2", "S1"), ("S3", "S2")):
    for model in ("lengkap", "cnn"):
        print(f"{a} - {b}, {'model lengkap' if model == 'lengkap' else 'CNN saja'}")
        display(M.bandingkan(ARCH, a, b, model=model).round(4))
F.fig_training_curves(ARCH); plt.show()


## XAI (pasca-pelatihan, kualitatif)
Dijalankan pada CNN final S1, S2, dan S3 dengan sampel yang dibekukan di sel 6:
1. **SHAP citra** (expected gradients) pada citra global dan crop lokal;
2. **SHAP fitur tabular berkelompok**;
3. **Kontribusi modalitas** (knock-out citra/tabular pada seluruh holdout).

Bila terputus, sampel yang sudah selesai disimpan dan dilewati. Batas klaim: pemeriksaan kualitatif fokus model, **bukan validasi klinis**, dan tidak dipakai untuk memeringkat model.


In [ ]:
import v4_xai as X
for scenario in ("S1", "S2", "S3"):
    receipt = X.explain(ARCH, scenario)
    print(scenario, "selesai | |residual additivity| median =", round(receipt["additivity_residual_abs_median"], 3))


In [ ]:
F.fig_xai_gallery(ARCH); plt.show()
segments = F.xai_segment_table(); display(segments[segments.Backbone == F.ARCH_NAMES[ARCH]])
modality = F.modality_table(); display(modality[modality.Backbone == F.ARCH_NAMES[ARCH]])
R.status(ARCH)
